In [1]:
import pandas as pd
import numpy as np

# ==========================================
# Load Dataset
# ==========================================

df = pd.read_csv("datasets/drilling_data_clean.csv")

print("="*60)
print("Original Shape:", df.shape)

# ==========================================
# Remove Duplicate Rows
# ==========================================

df.drop_duplicates(inplace=True)

# ==========================================
# Convert Time Column
# ==========================================

df["time"] = pd.to_datetime(df["time"], errors="coerce")

# Remove rows with invalid timestamps
df = df.dropna(subset=["time"])

# ==========================================
# Create Time Features
# ==========================================

df["year"] = df["time"].dt.year
df["month"] = df["time"].dt.month
df["day"] = df["time"].dt.day
df["hour"] = df["time"].dt.hour
df["day_of_week"] = df["time"].dt.dayofweek

# Drop original timestamp
df.drop(columns=["time"], inplace=True)

# ==========================================
# Drop Columns With >80% Missing Values
# ==========================================

missing_percent = df.isnull().mean()

cols_to_drop = missing_percent[missing_percent > 0.80].index

print("\nColumns Dropped (>80% Missing):")
print(list(cols_to_drop))

df.drop(columns=cols_to_drop, inplace=True)

# ==========================================
# Remove Constant Columns
# ==========================================

constant_cols = [c for c in df.columns if df[c].nunique() <= 1]

print("\nConstant Columns:")
print(constant_cols)

df.drop(columns=constant_cols, inplace=True)

# ==========================================
# Fill Missing Values
# ==========================================

numeric_cols = df.select_dtypes(include=np.number).columns

for col in numeric_cols:
    df[col] = df[col].fillna(df[col].median())

# ==========================================
# Optional: Remove Negative Values
# (Only for measurements that cannot be negative)
# ==========================================

non_negative_cols = [
    "weight_on_bit",
    "hookload",
    "rop_depth_hour",
    "top_drive_rpm",
    "flow_in",
    "pump_pressure",
    "spm_total",
    "pit_volume_active",
    "return_flow",
    "total_depth",
    "bit_rpm",
    "depth_hole_tvd",
    "downhole_torque"
]

for col in non_negative_cols:
    if col in df.columns:
        df.loc[df[col] < 0, col] = np.nan
        df[col] = df[col].fillna(df[col].median())

# ==========================================
# Remove Infinite Values
# ==========================================

df.replace([np.inf, -np.inf], np.nan, inplace=True)

for col in numeric_cols:
    if col in df.columns:
        df[col] = df[col].fillna(df[col].median())

# ==========================================
# Final Information
# ==========================================

print("="*60)
print("Final Shape:", df.shape)

print("\nRemaining Missing Values:")
print(df.isnull().sum().sort_values(ascending=False).head())

print("\nData Types:")
print(df.dtypes)

# ==========================================
# Save Clean Dataset
# ==========================================

df.to_csv("drilling_data_ml_ready.csv", index=False)

print("\nClean dataset saved as:")
print("drilling_data_ml_ready.csv")

Original Shape: (608676, 36)

Columns Dropped (>80% Missing):
['mwd_gamma_api', 'res_ps_2mhz_18in', 'res_ps_400khz_18in', 'rss_azimuth']

Constant Columns:
[]
Final Shape: (600359, 36)

Remaining Missing Values:
block_position    0
weight_on_bit     0
mwd_azimuth       0
mud_temp          0
h2s_01            0
dtype: int64

Data Types:
block_position             float64
weight_on_bit              float64
hookload                   float64
slips_set                  float64
rop_depth_hour             float64
on_bottom                  float64
top_drive_rpm              float64
top_drive_torque_ft_lbs    float64
flow_in                    float64
pump_pressure              float64
spm_total                  float64
pit_volume_active          float64
pit_gl_active              float64
gas_total_units            float64
trip_volume_active         float64
trip_gl                    float64
return_flow                float64
rig_mode                   float64
rockit_on_off              float